[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Transactions and Errors &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `parent` and `child` tables. Run it first. Each
task rebuilds those tables before it starts, so they can be run in any order.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def build_tables():
    """A parent and a child, so this notebook has both kinds of constraint to break."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS child, parent CASCADE")
        conn.execute("CREATE TABLE parent (id int PRIMARY KEY)")
        conn.execute("INSERT INTO parent VALUES (1)")
        conn.execute("CREATE TABLE child (id int PRIMARY KEY, "
                     "parent_id int REFERENCES parent (id))")


def ids():
    """Whatever is in child now, as a list, so a cell can show what survived."""
    with psycopg.connect("dbname=guide") as conn:
        return [row[0] for row in conn.execute("SELECT id FROM child ORDER BY id")]


print("server:", start_server())
print(report())
build_tables()
print("parent and child are ready | child holds:", ids())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
parent and child are ready | child holds: []


**1.** A connection before and after it is poisoned.


In [2]:
conn = psycopg.connect("dbname=guide")
conn.execute("SELECT 1")
print("a healthy transaction:", conn.info.transaction_status.name)

try:
    conn.execute("SELECT 1/0")
except errors.DivisionByZero:
    pass
print("after a failure:      ", conn.info.transaction_status.name)

conn.rollback()
print("after a rollback:     ", conn.info.transaction_status.name)
print("and it answers again: ", conn.execute("SELECT 1").fetchone())
conn.close()


a healthy transaction: INTRANS
after a failure:       INERROR
after a rollback:      IDLE
and it answers again:  (1,)


`INERROR` is the state that makes every later statement fail. A rollback is the only way out of it,
and afterwards the connection is as good as new.


**2.** What a violation carries.


In [3]:
build_tables()

with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("INSERT INTO child VALUES (1, 1)")
    try:
        conn.execute("INSERT INTO child VALUES (1, 1)")
    except errors.UniqueViolation as error:
        print("class:     ", type(error).__name__)
        print("sqlstate:  ", error.sqlstate)
        print("constraint:", error.diag.constraint_name)


class:      UniqueViolation
sqlstate:   23505
constraint: child_pkey


The class comes from the code, and the code comes from the server. The constraint name is the part
worth acting on: it says which rule was broken, which a message in English cannot be relied on to do.


**3.** Four rows, each in its own savepoint.


In [4]:
build_tables()

kept, skipped = [], []
with psycopg.connect("dbname=guide") as conn:
    for number in (10, 11, 11, 12):
        try:
            with conn.transaction():
                conn.execute("INSERT INTO child VALUES (%s, 1)", (number,))
            kept.append(number)
        except errors.UniqueViolation:
            skipped.append(number)

print("kept:   ", kept)
print("skipped:", skipped)
print("rows:   ", ids())


kept:    [10, 11, 12]
skipped: [11]
rows:    [10, 11, 12]


The savepoint is what keeps the duplicate from touching anything else. Each block is rolled back on
its own, and the transaction around them never enters the aborted state.


**4.** The same four rows, with nothing to catch them.


In [5]:
build_tables()

with psycopg.connect("dbname=guide") as conn:
    try:
        for number in (10, 11, 11, 12):
            conn.execute("INSERT INTO child VALUES (%s, 1)", (number,))
    except errors.UniqueViolation:
        print("stopped at the second 11")

print("rows:", ids(), "<- 10 and 11 were written before the failure, and are gone")


stopped at the second 11
rows: [] <- 10 and 11 were written before the failure, and are gone


Two rows were written and none survived. The duplicate aborted the transaction, and everything in it
went back, which is the whole argument for the savepoint in the task before this one.


**5.** Codes, and one with no class.


In [6]:
for code in ("23505", "23503"):
    print(f"  {code} -> {errors.lookup(code).__name__}")

try:
    errors.lookup("ZZ999")
except KeyError as error:
    print("  ZZ999 ->", type(error).__name__ + ":", error)


  23505 -> UniqueViolation
  23503 -> ForeignKeyViolation
  ZZ999 -> KeyError: 'ZZ999'


`lookup` is the way back from a code in a log to the class that would have been raised. A code
psycopg has no class for raises `KeyError`, which is worth catching if the codes are coming from
somewhere you do not control.


**6.** The same number, from the other driver.


In [7]:
build_tables()
conn = await asyncpg.connect(database="guide")
await conn.execute("INSERT INTO child VALUES (1, 1)")

try:
    await conn.execute("INSERT INTO child VALUES (1, 1)")
except asyncpg.exceptions.UniqueViolationError as error:
    print("asyncpg class:  ", type(error).__name__)
    print("asyncpg sqlstate:", error.sqlstate)
    print("psycopg's class for that code:", errors.lookup(error.sqlstate).__name__)
await conn.close()


asyncpg class:   UniqueViolationError
asyncpg sqlstate: 23505
psycopg's class for that code: UniqueViolation


Two drivers, two class names, one number. The code is PostgreSQL's, so it is the thing to match on
in any code that has to work with both, and `errors.lookup` turns it back into whichever name the
reader of that code expects.


---

&#8592; **Back to:** [Transactions and Errors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/03-transactions-and-errors.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
